In [1]:
# Imports and general variables

import pandas as pd 
import numpy as np
import requests
from io import StringIO

In [66]:
# Load the data from SCB API

url = "https://api.scb.se/OV0104/v1/doris/sv/ssd/START/BE/BE0401/BE0401A/BefProgRegFakN"

query = {
  "query": [
    {
      "code": "Region",
      "selection": {
        "filter": "vs:RegionLän07EjAggr",
        "values": ["01","03","04","05","06","07","08","09","10","12","13","14","17","18","19","20","21","22","23","24","25"]
      }
    },
    {
      "code": "InrikesUtrikes",
      "selection": {
        "filter": "item",
        "values": ["83"]
      }
    },
    {
      "code": "Kon",
      "selection": {
        "filter": "item",
        "values": ["1","2"]
      }
    },
    {
      "code": "Alder",
      "selection": {
        "filter": "agg:Ålder10årJ",
        "values": ["-9","10-19","20-29","30-39","40-49","50-59","60-69","70-79","80-89","90-99","100+"]
      }
    },
    {
      "code": "Tid",
      "selection": {
        "filter": "item",
        "values": ["2025","2026","2027","2028","2029","2030","2031","2032","2033","2034","2035","2036","2037","2038","2039","2040","2041","2042","2043","2044","2045"]
      }
    }
  ],
  "response": {
    "format": "csv"
  }
}

##  Make the request (POST)
response = requests.post(url, json=query)
if response.status_code == 200:
    csv_data = StringIO(response.text)
    response_csv = pd.read_csv(csv_data)    
else:
    print(f"Error: {response.status_code}")

## Format dataframe
population_data = response_csv.copy()

In [38]:
population_data

,region,inrikes/utrikes född,kön,ålder,Antal 2025,Antal 2026,Antal 2027,Antal 2028,Antal 2029,Antal 2030,...,Antal 2036,Antal 2037,Antal 2038,Antal 2039,Antal 2040,Antal 2041,Antal 2042,Antal 2043,Antal 2044,Antal 2045
0,01 Stockholms län,inrikes och utrikes födda,män,0-9 år,137322.0364,134060.8280,131510.0420,129251.5460,127323.4770,125708.2965,...,128851.9034,130646.0787,132439.1678,134222.9571,135971.8564,137582.5637,139017.7400,140236.0476,141391.1178,142437.2115
1,01 Stockholms län,inrikes och utrikes födda,män,10-19 år,152256.1908,151955.0926,150817.6949,149429.4355,147975.8147,146065.3759,...,131972.8971,129906.0434,128022.9489,126394.1422,125024.1503,123783.7315,123830.6387,124307.0147,125389.1118,126744.6133
2,01 Stockholms län,inrikes och utrikes födda,män,20-29 år,148761.4348,150152.8443,152015.3896,153771.2303,154511.6001,156568.6164,...,162912.2248,162348.5148,161619.8103,160829.5201,159613.6400,159071.3135,157469.1159,155649.3735,153372.8256,151025.7546
3,01 Stockholms län,inrikes och utrikes födda,män,30-39 år,200833.4781,199225.4430,197146.8730,194641.8687,192875.7445,189873.6342,...,183761.6504,185851.2060,187866.0145,189168.5023,191654.4262,193873.6101,195729.0391,197215.3173,198276.6291,199278.8776
4,01 Stockholms län,inrikes och utrikes födda,män,40-49 år,174875.2551,176736.1770,179054.5933,182236.3898,185086.7234,187399.6910,...,194946.9040,193495.9490,191696.0751,190382.3827,187936.7519,185336.4293,183322.4886,182232.6119,181721.2638,181883.2515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457,25 Norrbottens län,inrikes och utrikes födda,kvinnor,60-69 år,15551.9724,15454.2132,15473.6221,15467.0596,15295.5237,15206.4051,...,14149.4096,13717.4293,13327.9003,13123.5461,12894.5531,12714.4990,12457.3567,12243.2149,12110.5033,12058.4952
458,25 Norrbottens län,inrikes och utrikes födda,kvinnor,70-79 år,14519.3986,14427.8537,14272.1948,14002.3401,13924.9116,13857.8227,...,13704.9568,13758.9291,13780.1294,13661.4482,13612.8240,13582.1643,13596.1170,13546.0713,13370.5955,13139.7411
459,25 Norrbottens län,inrikes och utrikes födda,kvinnor,80-89 år,8735.8962,9056.0608,9338.4338,9647.5803,9769.9479,9893.5597,...,9963.1521,9920.4857,9802.5637,9802.8583,9800.4986,9760.3784,9728.8988,9721.8965,9848.7725,9962.7668
460,25 Norrbottens län,inrikes och utrikes födda,kvinnor,90-99 år,1836.8891,1873.5704,1925.5663,1959.5518,2041.0261,2067.4797,...,2575.6446,2690.9657,2807.9542,2849.6945,2868.5908,2874.3284,2889.8968,2907.0027,2898.3087,2907.2660


In [67]:
population_data['geography'] = population_data['region'].apply(lambda x: x.split(' ', 1)[0]) # Split the municipality code and the name
population_data.drop(columns=['inrikes/utrikes född', 'region'], inplace=True)
population_data.rename(columns={'kön': 'gender', 'ålder': 'age'}, inplace=True)
population_data = population_data.melt(
    id_vars=["gender", "age", "geography"],
    var_name="year",
    value_name="population"
)
population_data["year"] = population_data["year"].str.extract(r"(\d{4})").astype(int)
population_data["population"] = pd.to_numeric(population_data["population"], errors="coerce")
population_data = population_data.groupby(['geography', 'year', 'age']).sum().reset_index().drop(columns=['gender'])


In [68]:
population_data

,geography,year,age,population
0,01,2025,0-9 år,268245.6487
1,01,2025,10-19 år,294783.5158
2,01,2025,100+ år,649.2899
3,01,2025,20-29 år,291292.1422
4,01,2025,30-39 år,394958.6271
...,...,...,...,...
4846,25,2045,50-59 år,31178.9043
4847,25,2045,60-69 år,25080.7760
4848,25,2045,70-79 år,26336.9668
4849,25,2045,80-89 år,19133.9607


In [69]:
def map_age_group(age):
    if age in ["0-9 år", "10-19 år"]:
        return "0-19"
    elif age in ["20-29 år", "30-39 år", "40-49 år", "50-59 år", "60-69 år"]:
        return "20-69"
    else:
        return "70+"

# Apply mapping
population_data["age_group"] = population_data["age"].map(map_age_group)

# Group by geography, year, and new age group, summing population
population_data = population_data.groupby(["geography", "year", "age_group"], as_index=False)["population"].sum()


In [72]:
population_data.to_csv('county_population.csv', index=False)
